# VideoPainter - Official Environment

**Runtime:** A100 GPU

**This notebook uses the EXACT official dependencies (PyTorch 2.4.0, etc.).**

### Instructions
1. Run Cell 1 to install everything (will trigger runtime restart - this is NORMAL)
2. After restart, run Cell 2 onwards
3. Do NOT re-run Cell 1 after restart

## Cell 1: Install Official Environment (run ONCE, triggers restart)

In [ ]:
# Clone repo
import os
os.chdir('/content')
if not os.path.exists('/content/VideoPainter'):
    !git clone https://github.com/TencentARC/VideoPainter.git

# Install OFFICIAL PyTorch 2.4.0 (replaces Colab's default)
!pip install torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121 -q 2>&1 | tail -3

# Install official dependencies (skip DeepSpeed/pytorchvideo/gradio/CLIP to avoid issues)
!pip install transformers==4.42.2 accelerate>=0.34.2 -q 2>&1 | tail -2
!pip install numpy==1.26.0 scipy einops decord==0.6.0 -q 2>&1 | tail -2
!pip install imageio>=2.35.1 imageio-ffmpeg==0.5.1 -q 2>&1 | tail -2
!pip install safetensors==0.4.3 huggingface_hub==0.24.1 -q 2>&1 | tail -2
!pip install omegaconf==2.3.0 hydra-core iopath -q 2>&1 | tail -2
!pip install peft sentencepiece -q 2>&1 | tail -2

# Install custom diffusers PROPERLY via pip install -e (official way)
!cd /content/VideoPainter/diffusers && pip install -e . -q 2>&1 | tail -3

# Install SAM2
!cd /content/VideoPainter/app && pip install -e . -q 2>&1 | tail -3

print('\n=== Installation complete ===')
print('If Colab prompts to restart runtime, click RESTART.')
print('Then skip this cell and run Cell 2.')

---
## >>> After restart, start from here <<<
---
## Cell 2: Verify Environment

In [ ]:
import torch, diffusers, numpy as np
print(f'PyTorch: {torch.__version__}')   # Should be 2.4.0
print(f'CUDA: {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'diffusers: {diffusers.__version__}')  # Should be 0.31.0.dev0
print(f'numpy: {np.__version__}')        # Should be 1.26.0

from diffusers import CogVideoXDPMScheduler, CogvideoXBranchModel, CogVideoXI2VDualInpaintAnyLPipeline
print('Pipeline imports: OK')

from sam2.build_sam import build_sam2_video_predictor
print('SAM2: OK')

assert torch.__version__.startswith('2.4'), f'Wrong PyTorch: {torch.__version__}, expected 2.4.x'
print('\n=== Environment verified ===')

## Cell 3: Download Models from HuggingFace

In [ ]:
import os
os.chdir('/content/VideoPainter')
os.makedirs('ckpt', exist_ok=True)

from huggingface_hub import snapshot_download
import urllib.request

if not os.path.exists('ckpt/CogVideoX-5b-I2V/model_index.json'):
    print('[1/3] CogVideoX-5b-I2V (~21GB)...')
    snapshot_download(repo_id='THUDM/CogVideoX-5b-I2V', local_dir='ckpt/CogVideoX-5b-I2V')
else: print('[1/3] exists')

if not os.path.exists('ckpt/VideoPainter/checkpoints/branch/config.json'):
    print('[2/3] VideoPainter (~680MB)...')
    snapshot_download(repo_id='TencentARC/VideoPainter', local_dir='ckpt/VideoPainter')
else: print('[2/3] exists')

if not os.path.exists('ckpt/sam2_hiera_large.pt'):
    print('[3/3] SAM2 (~857MB)...')
    urllib.request.urlretrieve(
        'https://huggingface.co/facebook/sam2-hiera-large/resolve/main/sam2_hiera_large.pt',
        'ckpt/sam2_hiera_large.pt')
else: print('[3/3] exists')

# Fix ckpt path if HuggingFace adds extra nesting
branch_path = 'ckpt/VideoPainter/checkpoints/branch/config.json'
nested_path = 'ckpt/VideoPainter/VideoPainter/checkpoints/branch/config.json'
if not os.path.exists(branch_path) and os.path.exists(nested_path):
    BRANCH_DIR = './ckpt/VideoPainter/VideoPainter/checkpoints/branch'
    print(f'Note: branch at nested path {BRANCH_DIR}')
else:
    BRANCH_DIR = './ckpt/VideoPainter/checkpoints/branch'

os.environ['BRANCH_DIR'] = BRANCH_DIR
print(f'\nBranch: {BRANCH_DIR}')
print('Models ready!')

## Cell 4: Upload Video

In [ ]:
import os
os.chdir('/content/VideoPainter')
os.makedirs('test_videos', exist_ok=True)
from google.colab import files
uploaded = files.upload()
for fn in uploaded:
    dst = f'test_videos/{fn}'
    if os.path.exists(fn): os.rename(fn, dst)
    print(f'Saved: {dst}')
!ls -lh test_videos/

## Cell 5: Config

In [ ]:
# ========================================
#  Modify your parameters here
# ========================================
VIDEO_PATH = 'test_videos/0006-handbag1_scene01.mp4'
CLICK_X = 360
CLICK_Y = 240
PROMPT = 'A red leather wallet on a table'
NUM_STEPS = 50
GUIDANCE = 6.0
SEED = 42
DILATE = 32
OUTPUT_DIR = 'output_official'
# ========================================
print(f'{VIDEO_PATH} | ({CLICK_X},{CLICK_Y}) | {NUM_STEPS} steps')

## Cell 6: Run Full Pipeline (Official Logic)

This cell follows the exact same logic as the official `infer/inpaint.py`:
1. SAM2 segmentation + tracking
2. Load branch to CUDA, then create pipeline
3. `pipe.to('cuda')` (official style, no CPU offload)
4. Run inpainting with official parameters

In [ ]:
import os, sys, gc, torch, cv2, numpy as np
from PIL import Image
os.chdir('/content/VideoPainter')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============ Step 1: Extract Frames ============
print('[1/5] Loading video...')
from decord import VideoReader
vr = VideoReader(VIDEO_PATH)
fps_orig = vr.get_avg_fps()
total = len(vr)
dur = total / fps_orig
print(f'  {total} frames, {fps_orig:.1f} fps, {dur:.1f}s')

target_fps = 8
max_seconds = 6.0
use_frames = int(min(dur, max_seconds) * fps_orig)
step = max(1, int(fps_orig / target_fps))
indices = list(range(0, use_frames, step))
frames = np.array([cv2.resize(f, (720, 480)) for f in vr.get_batch(indices).asnumpy()])
print(f'  Extracted: {len(frames)} frames at {target_fps}fps')

# ============ Step 2: SAM2 Segmentation ============
print('[2/5] SAM2 segmentation...')
from sam2.build_sam import build_sam2_video_predictor
predictor = build_sam2_video_predictor('sam2_hiera_l.yaml', 'ckpt/sam2_hiera_large.pt')
state = predictor.init_state(images=frames, offload_video_to_cpu=True, async_loading_frames=True)
predictor.reset_state(state)
predictor.add_new_points(inference_state=state, frame_idx=0, obj_id=0,
    points=np.array([[CLICK_X, CLICK_Y]], dtype=np.float32),
    labels=np.array([1], dtype=np.int32))
masks = np.zeros((len(frames), 480, 720), dtype=np.uint8)
for fi, _, ml in predictor.propagate_in_video(state):
    masks[fi] = (ml[0, 0] > 0).cpu().numpy().astype(np.uint8)
print(f'  Mask: {masks.sum()} pixels in first frame')

# Save mask visualization
f0 = frames[0].copy()
cv2.circle(f0, (CLICK_X, CLICK_Y), 8, (0, 255, 0), -1)
ov = f0.copy(); ov[masks[0] > 0] = [255, 0, 0]
blend = cv2.addWeighted(f0, 0.6, ov, 0.4, 0)
cv2.imwrite(f'{OUTPUT_DIR}/frame0_with_mask.png', cv2.cvtColor(blend, cv2.COLOR_RGB2BGR))

# Free SAM2 GPU memory completely
del predictor, state; gc.collect(); torch.cuda.empty_cache()

# ============ Step 3: Prepare Data (Official Style) ============
print('[3/5] Preparing data...')
video_pil, masked_pil, mask_pil = [], [], []
for i in range(len(frames)):
    fr, mk = frames[i], masks[i]
    video_pil.append(Image.fromarray(fr).convert('RGB'))
    # Official: dilate only when img_inpainting_model is set. We dilate anyway for better edges.
    if DILATE > 0:
        dl = cv2.dilate(mk, np.ones((DILATE, DILATE), np.uint8))
    else:
        dl = mk
    # Masked frame (black out masked region) - same as official read_video_with_mask()
    mf = fr.copy(); mf[dl > 0] = 0
    masked_pil.append(Image.fromarray(mf).convert('RGB'))
    # Binary mask (255=inpaint region) - same as official
    bm = np.where(dl > 0, 255, 0).astype(np.uint8)
    mask_pil.append(Image.fromarray(bm).convert('RGB'))

# Trim to 4n+1 frames, max 49 (official: inpainting_frames=49)
inpainting_frames = 49
n_use = min(len(video_pil), inpainting_frames)
num_frames = ((n_use - 1) // 4) * 4 + 1
video_pil = video_pil[:num_frames]
masked_pil = masked_pil[:num_frames]
mask_pil = mask_pil[:num_frames]
print(f'  Using {num_frames} frames')

# First frame GT (official: --first_frame_gt)
gt_mask_first = mask_pil[0]
gt_vid_first = video_pil[0]
mask_pil[0] = Image.fromarray(np.zeros_like(np.array(mask_pil[0]))).convert('RGB')

# ============ Step 4: Load Pipeline (OFFICIAL STYLE) ============
print('[4/5] Loading models (official style)...')
from diffusers import CogVideoXDPMScheduler, CogvideoXBranchModel, CogVideoXI2VDualInpaintAnyLPipeline
from diffusers.utils import export_to_video

BRANCH_DIR = os.environ.get('BRANCH_DIR', './ckpt/VideoPainter/checkpoints/branch')
dtype = torch.bfloat16

# Official: branch.cuda() BEFORE pipeline creation
print('  Loading branch...')
branch = CogvideoXBranchModel.from_pretrained(BRANCH_DIR, torch_dtype=dtype).to(dtype=dtype).cuda()

print('  Loading pipeline...')
pipe = CogVideoXI2VDualInpaintAnyLPipeline.from_pretrained(
    'ckpt/CogVideoX-5b-I2V', branch=branch, torch_dtype=dtype)
pipe.text_encoder.requires_grad_(False)
pipe.transformer.requires_grad_(False)
pipe.vae.requires_grad_(False)
pipe.branch.requires_grad_(False)

# Official: scheduler with trailing
pipe.scheduler = CogVideoXDPMScheduler.from_config(pipe.scheduler.config, timestep_spacing='trailing')

# Official: pipe.to('cuda'), NO slicing/tiling for non-long-video
pipe.to('cuda')
print(f'  GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB used')

# ============ Step 5: Run Inpainting (OFFICIAL STYLE) ============
print(f'[5/5] Running inpainting ({num_frames} frames, {NUM_STEPS} steps)...')
image = masked_pil[0]  # Official: image = masked_video[0]

inpaint_outputs = pipe(
    prompt=PROMPT,
    image=image,
    num_videos_per_prompt=1,
    num_inference_steps=NUM_STEPS,
    num_frames=num_frames,
    use_dynamic_cfg=True,
    guidance_scale=GUIDANCE,
    generator=torch.Generator().manual_seed(SEED),
    video=masked_pil,
    masks=mask_pil,
    strength=1.0,
    replace_gt=True,        # Official: --replace_gt
    mask_add=True,          # Official: --mask_add
    stride=num_frames,      # Official: stride = frames - overlap_frames (overlap=0)
    prev_clip_weight=0.0,   # Official: prev_clip_weight=0.0
    output_type='np'
).frames[0]

print('Inference complete!')

# ============ Save Results ============
mask_pil[0] = gt_mask_first
video_pil[0] = gt_vid_first
export_to_video(list(inpaint_outputs), f'{OUTPUT_DIR}/result.mp4', fps=8)

# Build visualization (same as official _visualize_video)
orig_t = pipe.video_processor.preprocess_video(video_pil[:len(inpaint_outputs)],
    height=inpaint_outputs.shape[1], width=inpaint_outputs.shape[2])
mask_t = pipe.masked_video_processor.preprocess_video(mask_pil[:len(inpaint_outputs)],
    height=inpaint_outputs.shape[1], width=inpaint_outputs.shape[2])
masked_t = orig_t * (mask_t < 0.5)
orig_np = pipe.video_processor.postprocess_video(video=orig_t, output_type='np')[0]
masked_np = pipe.video_processor.postprocess_video(video=masked_t, output_type='np')[0]
mask_np = mask_t.squeeze(0).squeeze(0).numpy()[..., np.newaxis].repeat(3, axis=-1)

vis = [np.concatenate([orig_np[i], masked_np[i], mask_np[i], inpaint_outputs[i]], axis=1)
       for i in range(min(len(inpaint_outputs), len(orig_np)))]
export_to_video(vis, f'{OUTPUT_DIR}/result_vis.mp4', fps=8)

# Extract key frames
for idx, name in [(0, 'first'), (len(inpaint_outputs)//2, 'mid'), (len(inpaint_outputs)-1, 'last')]:
    cv2.imwrite(f'{OUTPUT_DIR}/frame_{name}.png',
        cv2.cvtColor((inpaint_outputs[idx]*255).astype(np.uint8), cv2.COLOR_RGB2BGR))

print(f'\nResults saved to {OUTPUT_DIR}/')
print('Done!')

## Cell 7: View Results

In [ ]:
import cv2
from IPython.display import display, Image as I, Video as V

# Show mask
print('=== Segmentation ===')
display(I(filename=f'{OUTPUT_DIR}/frame0_with_mask.png', width=720))

# Show vis frames
print('\n=== Results (Original | Masked | Mask | Generated) ===')
for src in ['result_vis']:
    cap = cv2.VideoCapture(f'{OUTPUT_DIR}/{src}.mp4')
    tot = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    for nm, ix in [('vis_first',0),('vis_mid',tot//2),('vis_last',tot-1)]:
        cap.set(cv2.CAP_PROP_POS_FRAMES, ix)
        r, f = cap.read()
        if r: cv2.imwrite(f'{OUTPUT_DIR}/{nm}.png', f)
    cap.release()

for lb, fn in [('First','vis_first'),('Mid','vis_mid'),('Last','vis_last')]:
    print(f'--- {lb} ---')
    display(I(filename=f'{OUTPUT_DIR}/{fn}.png', width=1000))

print('\n=== Video ===')
display(V(f'{OUTPUT_DIR}/result.mp4', embed=True, width=720))

## Cell 8: Download Results

In [ ]:
import shutil
shutil.make_archive(f'/content/{OUTPUT_DIR}', 'zip', '.', OUTPUT_DIR)
from google.colab import files
files.download(f'/content/{OUTPUT_DIR}.zip')